# 18 - Qwen3-Embedding-8B LoRA Fine-tuning and Evaluation

Fine-tune `Qwen/Qwen3-Embedding-8B` with leakage-checked legal contrastive pairs, rebuild the official-law index, and evaluate retrieval on the locked benchmark.


In [ ]:
!pip install -q -U "sentence-transformers>=2.7.0" "transformers>=4.51.0" accelerate peft bitsandbytes faiss-cpu rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))
config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from src.finetune_embedding_model import finetune_embedding_model

tuned_model_dir = DRIVE_ROOT / 'models/embedding_tuned/qwen3_embedding_8b_legal_lora_v1'
run_config = finetune_embedding_model(
    train_jsonl=DRIVE_ROOT / 'data/processed/embedding_tune_train.jsonl',
    val_jsonl=DRIVE_ROOT / 'data/processed/embedding_tune_val.jsonl',
    output_dir=tuned_model_dir,
    model_name='Qwen/Qwen3-Embedding-8B',
    max_train_samples=None,
    max_val_samples=None,
    batch_size=1,
    epochs=1,
    use_hard_negative=True,
    max_seq_length=1024,
    use_lora=True,
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    torch_dtype='bfloat16',
    device=device,
)
run_config


In [ ]:
from src.build_index import build_indexes

tuned_index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b_legal_tuned'
manifest = build_indexes(
    corpus_path=DRIVE_ROOT / config['main_law_corpus_csv'],
    index_root=tuned_index_root,
    embedding_model=str(tuned_model_dir),
    text_field=config['retrieval_text_field'],
    batch_size=8,
    device=device,
    build_dense=True,
    build_bm25=False,
)
manifest


In [ ]:
from src.evaluation_retrieval import evaluate_retrieval
summary = evaluate_retrieval(
    benchmark_csv=DRIVE_ROOT / config['benchmark_csv'],
    index_root=tuned_index_root,
    output_predictions_csv=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_legal_tuned_dense_predictions_v1.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_legal_tuned_dense_summary_v1.json',
    mode='dense',
    top_k=30,
    candidate_k=30,
    device=device,
)
summary
